In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from IPython.display import Image, display  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    load_joined_condition_wavelets,
    resolve_notebook_wavelet_cache_dir,
    resolve_wavelet_dir,
)
from src.analysis import assr_trials as at  # noqa: E402
from src.analysis import iva_quality  # noqa: E402
from src.analysis.iva_condition_comparison import (  # noqa: E402
    apply_component_signs,
)
from src.analysis.wavelet_ica import (  # noqa: E402
    align_iva_component_signs,
    iva_component_patterns,
    zscore_by_time,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    CoordinateSystems,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)

from src.io.loading import assr_electrode_mask  # noqa: E402
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_tf_maps,
    plot_condition_mean_topomaps,
    plot_participant_condition_tf_maps,
    plot_participant_condition_topomaps,
)
from src.visualization.iva_quality_plots import topo_info_subset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# IVA Decomposition of Wavelet Power — Channel Components, Both Conditions Pooled

Runs the **channel-as-independent** IVA-G workflow of
[`wavelet_iva_channel.ipynb`](../05-wavelet-iva-analysis/wavelet_iva_channel.ipynb)
unchanged, but on the **joined** condition
([`ConditionVariants.JOINED`](../../src/definitions/fields.py)) instead of one
condition at a time: Placebo and Psilocybin are pooled on the **subject axis**, so
every *recording* is one subject and each participant appears twice — once per
condition.

**Scope.** It loads, decomposes, resolves the per-recording component sign, and then
draws the Placebo/Psilocybin comparison of the component **channel topographies** and
**time-frequency maps** — over the whole recording and, for a stimulus experiment,
**averaged over stimuli** — as condition means and per participant. It runs no component
ranking and no synchrony analysis, and writes nothing to disk beyond the figures and the
reusable wavelet subset cache. The in-memory arrays are the input for whatever further
comparison comes next.

## Why the subject axis

IVA-G ties the **sample** axis (the source component vector, SCV) across datasets
and leaves the **mixing** axis free per dataset. Pooling the conditions as extra
subjects therefore means:

```
datasets = recordings   ← participant × condition (2P entries)
samples  = time × frequency   ← the axis IVA aligns across ALL recordings (SCV)
mixing   = channel            ← free per recording
```

So the recovered spectro-temporal sources `(F, T)` are shared by **both**
conditions — one aligned set of sources describes the whole pooled cohort — while
each recording keeps its own channel topography `(C,)`. A participant's Placebo and
Psilocybin recordings get *different* mixing matrices, which is exactly what makes a
per-condition contrast of the patterns meaningful afterwards.

> Contrast this with the **time-axis** join
> ([`ConditionVariants.JOINED_TRACKS`](../../src/definitions/fields.py), via
> `load_paired_condition_wavelets`), where each participant is one subject carrying
> both tracks end to end. There the mixing is estimated once over the whole
> recording, so the components are identical between conditions by construction and
> the contrast lives entirely in the sources. Both joins are implemented in
> [`src/analysis/condition_tracks.py`](../../src/analysis/condition_tracks.py).

## How the data is assembled

`load_joined_condition_wavelets` loads **each condition exactly as a
single-condition workflow does**, reusing the existing per-condition wavelet caches,
then stacks the participant-matched recordings on the subject axis
(`pool_condition_subjects`). Nothing upstream is re-run, re-aligned or
re-transformed, and no `Joined_*` cache is written — the pooled tensor exists only
in memory.

This works because the stimulus alignment was fitted over **every recording of both
conditions**, so the two conditions already share one time base and can be stacked
directly. Only participants with a recording in *both* conditions are kept, so the
subject axis is a balanced within-participant design.

## Reshape

```
Input:   (n_subjects, n_channels, n_freqs, n_times)   with n_subjects = 2 × n_pairs
Per-subject reshape:
         (n_channels,  n_freqs × n_times)
           ── mixing ──   ─── samples ───
Per-subject PCA over channels → N_PCA
Stack:   (N_PCA, n_freqs × n_times, n_subjects)  — IVA layout (N, T, K)
```

IVA-G requires a **square** mixing matrix per dataset, so each recording's
`(C, F·T)` matrix is reduced with its own **PCA over channels** to `N_PCA`
components before stacking.

## Variables produced

| Variable | Shape | Description |
|----------|-------|-------------|
| `pooled` | — | `PooledConditionSubjects` — the pooled dataset plus its per-subject participant/condition bookkeeping |
| `bb_data` | `(S, C, F, T)` | Raw 4-D pooled wavelet power tensor, `S = 2 × n_pairs` |
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power tensor |
| `X_subjects` | `(S, C, F·T)` | Per-subject z-scored reshape (mixing=C, samples=F·T) |
| `pcas` | list[`PCA`] | Per-subject fitted PCA over channels |
| `X_pca` | `(N_PCA, F·T, S)` | PCA-reduced IVA input layout |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing matrices (sign-aligned) |
| `sigma_corr` | `(N_PCA, S, S)` | Per-SCV cross-subject correlation matrix |
| `iva_sources` | `(S, N_PCA, F, T)` | Spectro-temporal sources reshaped to (freq, time) |
| `iva_time` | `(S, N_PCA, T)` | Frequency-collapsed temporal source (signed) |
| `iva_freq` | `(S, N_PCA, F)` | Time-collapsed spectral source (signed, ≈ 0) |
| `iva_components` | `(S, N_PCA, C)` | Per-subject channel topographies (mixing) |

Every one of these is on the **pooled** subject axis. Split any of them back into
the two conditions with `pooled.condition_subjects(array, condition)`, which
returns participant-matched rows.

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.ASSR
# The pooled dataset IS the condition here — the two real conditions below are
# stacked on the subject axis and the product is labelled Joined_<MusicType>.
CONDITION = ConditionVariants.JOINED
# Subject-axis block order: Placebo subjects first, then Psilocybin.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers. It must
# match the grid the cache was written with, or the cache is missed and recomputed.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

# ── Reuse / compute ──────────────────────────────────────────
# True is the intended setting: this notebook is built around reusing the two
# per-condition caches. False would recompute both from RAW_AFTER_ICA.
REUSE_WAVELETS = True

# ── Cohort, channel and time subset ────────────────────────────
# N_PAIRS_SUBSET counts *participants*, not subjects: each kept participant brings
# one recording per condition, so the pooled subject axis is 2 × N_PAIRS_SUBSET.
# Never slice the pooled subject axis by position — the leading rows are one whole
# condition block. Use ``pooled.select_participants`` (done in the loading cell).
N_PAIRS_SUBSET: int | None = 5
# NOTE: the sample axis here is F·T, i.e. ``n_freqs`` (=50) times larger than the
# time-only variants, and the subject axis is twice a single condition's. Keep
# N_TIMES_SUBSET modest during exploration so the (N_PCA, F·T, S) IVA input stays in
# memory; raise it (or set to None) for the full run on a node with more RAM.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 3000  # first N time samples (F·T = 50 * this)

# ── Comparison figures ────────────────────────────────────────
SAVE_PLOTS = True
# Which components the comparison figures cover. None = every component.
COMPONENTS_TO_PLOT: list[int] | None = None
# Reference lines on the TF panels. The ASSR is continuous 40 Hz stimulation, so the
# stimulation frequency is the row worth locating; onsets are drawn only when there are
# few enough not to wash the map out.
TF_FREQ_MARKS: list[float] = [40.0]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── Stimulus-locked epoch (Step 8) ────────────────────────────────────
# The paradigm window, taken from src.definitions.constants.AssrEpoch so this notebook,
# the 05 quality notebook and the headless run_wavelet_iva_channel.py --quality path all
# cut the SAME epoch: a short pre-onset baseline, then the stimulus plus an equally long
# post-stimulus interval. iva_quality.onset_window caps the post-onset span by the
# shortest inter-onset gap, so an epoch can never reach the next stimulus.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S  # short interval before each stimulus
EPOCH_POST_S = AssrEpoch.POST_ONSET_S  # stimulus + post-stimulus
# Minimum onsets that must fit the (possibly time-subset) window for the average to be
# worth drawing.
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── IVA settings ──────────────────────────────────────────────
# N_COMPONENTS_PCA reduces the *channel* axis here, so it must be <= n_channels
# (<= N_CHANNELS_SUBSET when set). 10 is the standard ASSR setting for this variant
# and the default of scripts/run_wavelet_iva_channel.py.
N_COMPONENTS_PCA = 10  # per-subject PCA dim over channels (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = True
IVA_RANDOM_STATE = 42  # seeds per-subject PCA + W_init

# ── Wavelet cache directories ─────────────────────────────────
# Two caches, and the difference matters for how long this notebook takes to start.
#
# WAVELET_DIR is the *source of truth*: <data>/processed/<experiment>/wavelets, one
# compressed file per condition spanning the full cohort, all 195 channels and the
# whole aligned time axis (tens of GB each for ASSR). Reading it means decompressing
# all of it, even to take a 32-channel slice.
#
# WAVELET_SUBSET_CACHE_DIR is the *per-extent* cache, under the stage-03 notebook that
# owns the Morlet transform. The first run at a given extent writes the trimmed tensor
# there; every later run reads it back and never opens the big cache. It is keyed by
# condition, frequency grid and extent — not by the notebook that wrote it — so the
# entries this notebook fills in are the same ones the 03/04/05 workflows read, and
# vice versa. Pass it to any wavelet workflow via ``subset_cache_dir``.
WAVELET_DIR: Path = resolve_wavelet_dir(None, EXPERIMENT_NAME) / "broadband"
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME) / "broadband"
)
# False recomputes from WAVELET_DIR and overwrites the subset cache — the escape hatch
# if the source-of-truth cache was rebuilt under the same extent.
REUSE_WAVELET_SUBSET_CACHE = True

# ── Plots directory ───────────────────────────────────────────
# Canonical notebook layout: plots/<experiment>/<broadband|bands>/<analysis_type>/,
# with pca_<n> isolating sweeps over N_COMPONENTS_PCA.
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "iva_channel_joined"
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment / group  : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Conditions pooled   : {[c.value for c in CONDITIONS_TO_POOL]} (subject axis)")
print(f"Wavelet source cache: {WAVELET_DIR}")
print(
    f"Wavelet subset cache: {WAVELET_SUBSET_CACHE_DIR}  "
    f"(reuse: {REUSE_WAVELET_SUBSET_CACHE})"
)
print(
    f"Frequencies         : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Per-subject PCA dim : {N_COMPONENTS_PCA}  (over channels)")
print(f"IVA optimisation    : {IVA_OPT_APPROACH}  (max_iter={IVA_MAX_ITER})")
print(f"Plots directory     : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")
print(
    f"Stimulus epoch      : [-{EPOCH_PRE_S}, {EPOCH_POST_S}] s around each onset "
    f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
    f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)"
)

## Data Loading — both conditions on one subject axis

One call does the whole join: load each condition's wavelet power from its existing
cache, keep the participants present in both, and stack them on the subject axis.

**First run vs. every run after.** The source-of-truth wavelet tensor is
decompressed in full before it is trimmed to the requested channel/time extent (the
stimulus alignment is defined over the whole cohort, so that cache cannot be written
per subset) — tens of GB per condition for ASSR. The two conditions are loaded and
trimmed **one at a time**, so the peak is one condition's cache, the same peak as a
single-condition notebook, not both at once. Run the first pass on a node sized for
that.

The trimmed result is then written to `WAVELET_SUBSET_CACHE_DIR`, and subsequent runs
at the same extent read it back without opening the big cache at all — which is both
much faster and much smaller in memory. Changing `N_CHANNELS_SUBSET` or
`N_TIMES_SUBSET` asks for a different extent and so pays the full cost once more, for
that extent. The cache is shared across workflows: whatever the 03/04/05 notebooks
have already stored for these conditions at this extent, this notebook reads.

In [ ]:
pooled, condition_analyzers = load_joined_condition_wavelets(
    MUSIC_TYPE,
    EXCLUSION_CATEGORIES,
    FREQS,
    wavelet_dir=WAVELET_DIR,
    experiment_name=EXPERIMENT_NAME,
    representation=REPRESENTATION,
    conditions=CONDITIONS_TO_POOL,
    n_channels=N_CHANNELS_SUBSET,
    n_times=N_TIMES_SUBSET,
    reuse_wavelets=REUSE_WAVELETS,
    subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
    reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
)

print(f"Pooled dataset : {pooled.data.label}")
print(f"Participants   : {pooled.n_pairs}  ->  {pooled.n_subjects} subjects")
print(f"Shape          : {pooled.data.data.shape}  (subjects × channels × freqs × times)")

# Restrict the cohort by *participant*, so both of a participant's recordings are
# kept and the pooled axis stays balanced.
if N_PAIRS_SUBSET is not None:
    keep = sorted(set(pooled.participants))[:N_PAIRS_SUBSET]
    pooled = pooled.select_participants(keep)
    print(f"\nUsing first {len(keep)} participant(s): {keep}")
    print(f"Shape          : {pooled.data.data.shape}")

print(f"\nSubject axis   : {list(pooled.subject_labels)}")

## Dataset Selection

There is only one dataset here — the pooled one — so this cell just unpacks it into
the same names the single-condition notebooks use (`bb_data`, `sfreq`, `n_subjects`,
…) and records the per-subject condition bookkeeping the pooled layout needs.

In [ ]:
LABEL = pooled.data.label

bb_ad = pooled.data
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

# Per-subject bookkeeping: which participant and which condition each row is.
subject_participants = list(pooled.participants)
subject_conditions = [c.value for c in pooled.subject_conditions]
subject_labels = list(pooled.subject_labels)
# Boolean subject-axis masks, the safe way to address one condition's half.
condition_masks = {c: pooled.condition_mask(c) for c in pooled.conditions}
# Partner row of the same participant's other condition, for paired read-outs later.
partner_index = pooled.partner_index

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
for condition, mask in condition_masks.items():
    print(f"  {condition.value:<12}: {int(mask.sum())} subject(s) at rows {np.flatnonzero(mask).tolist()}")
print(f"Partner rows : {partner_index.tolist()}")

# ── Stimulus onset markers ────────────────────────────────────
# Both conditions share the aligned time base, so one set of onsets describes every
# subject. They can differ by a sample or two of rounding between conditions, so the
# first pooled condition's onsets are taken as the reference. ``None`` for
# experiments without stimulus annotations (e.g. PSILO_MUSIC) -> no markers.
_onset_samples = pooled.condition_onsets(pooled.conditions[0])
if _onset_samples is None:
    stimulus_onset_times = np.array([])
else:
    # Keep only onsets inside the (possibly time-subset) window.
    stimulus_onset_times = _onset_samples[_onset_samples < n_times] / sfreq
print(f"Stimulus onsets in window : {len(stimulus_onset_times)}")

---
## Step 1 — Z-score and Per-Subject Reshape

Identical to the single-condition notebook, and deliberately so: **z-scoring**
normalises each `(subject, channel, frequency)` time series to zero mean and unit
variance over time. Wavelet power is ~1/f, so without it the low-frequency,
high-power samples would dominate the SCV covariance and "shared spectrum" would
collapse to "shared low frequencies".

Because every series is normalised **independently per subject**, this is also why
the pooling itself does no standardising — applying it before or after the
subject-axis stack gives the identical array. The consequence is worth stating
plainly: an overall power difference between the two conditions is normalised away
here, so what the pooled decomposition compares is temporal and spectral
*structure*, not amplitude.

**Reshape** is done independently per subject: each `(C, F, T)` slice becomes a
`(C, F·T)` matrix — **channels** form the mixing axis and the joint
**(frequency, time)** axis forms the samples. The flatten keeps frequency slow and
time fast (`index = f·T + t`), so the sample axis reshapes cleanly back to `(F, T)`
after IVA.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power |
| `X_subjects` | `(S, C, F·T)` | Per-subject z-scored reshape (mixing=C, samples=F·T) |

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Per-subject reshape: (S, C, F, T) → (S, C, F*T). bb_z is C-contiguous, so the
# flattened sample axis runs frequency-slow, time-fast (index = f*T + t) and
# reshapes cleanly back to (F, T) after IVA.
n_samples_ft = n_freqs * n_times
X_subjects = bb_z.reshape(n_subjects, n_channels, n_samples_ft)  # (S, C, F*T)

print(f"Per-subject reshape    : {X_subjects.shape}  (subjects, C, F*T)")
print(f"  Mixing dim (channels): {n_channels}")
print(f"  Samples per subject  : {n_samples_ft}  (F={n_freqs} * T={n_times})")
print(f"  Datasets (recordings): {n_subjects}  "
      f"({pooled.n_pairs} participants x {len(pooled.conditions)} conditions)")

---
## Step 2 — Per-Subject PCA Over Channels

`iva_g` assumes a **square** mixing matrix per dataset. Here the mixing axis is
**channels**, so each recording's `(C, F·T)` matrix is reduced with its **own PCA
over channels** to `N_PCA` channel-mixtures, after which the K recording matrices
are stacked into IVA's `(N, T, K)` layout.

Each of a participant's two recordings gets its **own** PCA — they are separate
datasets as far as IVA is concerned. `N_PCA` must be `<= n_channels`.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pcas[k]` | — | Fitted PCA object for subject `k` (over channels) |
| `pca_evr` | `(S, N_PCA)` | Explained-variance ratio per subject |
| `X_pca` | `(N_PCA, F·T, S)` | IVA input layout (N, T, K) |

In [ ]:
if N_COMPONENTS_PCA > n_channels:
    raise ValueError(
        f"N_COMPONENTS_PCA ({N_COMPONENTS_PCA}) must be <= n_channels "
        f"({n_channels}); PCA reduces the channel axis in this notebook."
    )

# Per-subject PCA over channels. PCA expects (n_samples, n_features); each
# subject's matrix is (C, F*T) → transpose to (F*T, C), fit, transform back to
# (F*T, N_PCA), transpose to (N_PCA, F*T), then stack along axis 2 for IVA.
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
pca_evr = np.zeros((n_subjects, N_COMPONENTS_PCA))

for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (F*T, C) — samples x channels for sklearn
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)  # (F*T, N_PCA)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T  # (N_PCA, F*T)
    pca_evr[k] = pca.explained_variance_ratio_
    print(
        f"  {subject_labels[k]:<18}: explained variance = "
        f"{pca_evr[k].sum() * 100:5.1f}% ({N_COMPONENTS_PCA} channel comps)"
    )

# IVA expects (N, T, K)
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

print(f"\nIVA input shape        : {X_pca.shape}  (N_PCA, F*T, K=subjects)")

# Sanity check on the reduction, coloured by condition: a recording whose channel
# PCA retains far less variance than the rest is worth knowing about before IVA.
_palette = dict(zip(pooled.conditions, sns.color_palette("muted", len(pooled.conditions))))
fig, ax = plt.subplots(figsize=(8, 4))
for k in range(n_subjects):
    ax.plot(
        np.arange(1, N_COMPONENTS_PCA + 1),
        np.cumsum(pca_evr[k]),
        marker="o",
        markersize=3,
        color=_palette[pooled.subject_conditions[k]],
        alpha=0.8,
        label=(
            pooled.subject_conditions[k].value
            if k == int(np.flatnonzero(condition_masks[pooled.subject_conditions[k]])[0])
            else None
        ),
    )
ax.set_xlabel("Number of PCA components (channels)")
ax.set_ylabel("Cumulative variance explained")
ax.set_title(f"Per-Subject Channel PCA — {LABEL}")
ax.axhline(0.9, ls="--", lw=0.6, color="gray")
ax.legend(fontsize=8, title="Condition")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 3 — Run IVA-G

`iva_g` returns a demixing matrix `W` of shape `(N, N, K)`. The spectro-temporal
scores for recording `k` are

```
S_pca[k]      = W[:, :, k] @ X_pca[:, :, k]                                # (N_PCA, F·T)
patterns[k]   = iva_component_patterns(W[:, :, k], pcas[k].components_)    # (N_PCA, C)
```

IVA's permutation ambiguity is **shared across datasets**, so the kth source in
recording 0 is the kth source in recordings `1..K-1` — no post-hoc matching. With
both conditions in the dataset list, that alignment now spans the conditions too:
component `k` means the same thing in a Placebo recording and in a Psilocybin one.

`Sigma_N[:, :, k]` is the kth SCV's covariance across **all** `K = 2P` recordings,
so its off-diagonal blocks already contain the within-condition and
between-condition couplings — the raw material for a later contrast. Nothing is read
out of it here beyond the sign alignment.

**Topographies are the forward (mixing) patterns**, i.e.
`pinv(W_k @ pcas[k].components_)` — *not* the unmixing rows. `iva_g` folds its
internal whitening `V_k` into the returned `W_k`, so the unmixing rows carry an
extra `Σ⁻¹` weighting that up-weights the lowest-variance retained PCA directions;
the filter and the pattern of the same component can be nearly uncorrelated.
`iva_component_patterns` does the inversion (same convention as MNE's
`ica.get_components()`).

| Returned by `iva_g` | Shape | Description |
|---------------------|-------|-------------|
| `W` | `(N_PCA, N_PCA, S)` | Per-subject demixing matrix |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `Sigma_N` | `(S, S, N_PCA)` | Per-SCV covariance across recordings (over the F·T axis) |
| `isi` | `float` | Joint ISI (only when ground-truth `A` is given) |

In [ ]:
# Deterministic W initialisation: random matrix per subject seeded by
# IVA_RANDOM_STATE.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))

W, cost, Sigma_N, isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

print(f"\nW shape           : {W.shape}  (N_PCA, N_PCA, K=subjects)")
print(f"Sigma_N shape     : {Sigma_N.shape}  (K, K, N_PCA)")
print(f"Iterations        : {len(cost)}")
print(f"Final cost        : {cost[-1]:.6f}")
if len(cost) >= IVA_MAX_ITER:
    print(f"  WARNING: hit max_iter={IVA_MAX_ITER}; W may not have converged "
          f"(W_diff_stop={IVA_W_DIFF_STOP}).")

# Cost-curve sanity check.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cost, marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("IVA cost")
ax.set_title(f"IVA-G Convergence — {LABEL}")
fig.tight_layout()
plt.show()
plt.close("all")

In [ ]:
# Resolve IVA's per-subject sign ambiguity. IVA recovers each component (SCV) only
# up to a per-subject sign, so subject i's copy of component k may be the negation of
# subject j's — which makes genuinely shared activity look anti-correlated. Left
# unresolved on a pooled dataset it would also fabricate condition differences, since
# the flips fall arbitrarily across the two blocks. For each component we normalise
# Sigma_N to a subject x subject correlation matrix, take its leading eigenvector (the
# dominant cross-subject direction, oriented so its largest-magnitude entry is
# positive), and flip every subject whose loading on it is negative. Flips are applied
# to both W and the correlation stack (sigma_corr), so every quantity recovered from W
# below is sign-aligned across every recording of both conditions.
sigma_corr, W, sign_flips = align_iva_component_signs(Sigma_N, W)
n_flipped = int((sign_flips < 0).sum())
print(
    f"Sign alignment: flipped {n_flipped} (component, subject) pairs "
    f"across {N_COMPONENTS_PCA} components."
)
for condition, mask in condition_masks.items():
    print(f"  {condition.value:<12}: {int((sign_flips[:, mask] < 0).sum())} flip(s)")
print(f"sigma_corr shape  : {sigma_corr.shape}  (N_PCA, K, K)")
print(f"W shape           : {W.shape}  (N_PCA, N_PCA, K) — sign-aligned")

---
## Step 4 — Recover Spectro-Temporal Sources and Channel Patterns

Apply each recording's demixing matrix to obtain the IVA scores on the aligned `F·T`
axis, reshape them back to `(F, T)` spectro-temporal **sources**, and combine `W_k`
with the PCA loadings to express each component's **channel topography** `(C,)`.

We also store two marginals of the source:

- `iva_time` — frequency-collapsed temporal source, signed `(S, N_PCA, T)`
- `iva_freq` — time-collapsed spectral source, signed (≈ 0 by construction) `(S, N_PCA, F)`

The kth source/topography is **already aligned** across every recording, both
conditions included. Row `s` of each array belongs to `subject_labels[s]`; split them
by condition with `pooled.condition_subjects(array, condition)`.

In [ ]:
iva_scores_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_channels))

for k in range(n_subjects):
    W_k = W[:, :, k]  # (N_PCA, N_PCA)
    X_pca_k = X_pca[:, :, k]  # (N_PCA, F*T)

    # Spectro-temporal scores on the aligned sample axis: (N_PCA, F*T)
    iva_scores_pca[k] = W_k @ X_pca_k

    # Forward (mixing) patterns in channel space: (N_PCA, C). NOT the
    # unmixing rows — after iva_g's internal whitening those carry a
    # Σ⁻¹ reweighting of the channel-PCA directions.
    iva_components[k] = iva_component_patterns(W_k, pcas[k].components_)

# Reshape the aligned F*T axis back to (F, T): the shared spectro-temporal
# source. Flatten was frequency-slow, time-fast, so this is exact.
iva_sources = iva_scores_pca.reshape(
    n_subjects, N_COMPONENTS_PCA, n_freqs, n_times
)  # (S, N_PCA, F, T)

# Signed marginals over the collapsed axis — signs preserved for the diverging
# (RdBu_r) plots downstream. NB: iva_freq (mean over time) is ~0 by construction
# because zscore_by_time zeroes each (s, c, f) series' time-mean; kept signed.
iva_time = iva_sources.mean(axis=2)  # (S, N_PCA, T) — frequency-collapsed (signed)
iva_freq = iva_sources.mean(axis=3)  # (S, N_PCA, F) — time-collapsed (signed)

print(f"IVA scores (F*T)       : {iva_scores_pca.shape}  (S, N_PCA, F*T)")
print(f"IVA sources (F, T)     : {iva_sources.shape}  (S, N_PCA, F, T)")
print(f"IVA temporal marginal  : {iva_time.shape}  (S, N_PCA, T)")
print(f"IVA spectral marginal  : {iva_freq.shape}  (S, N_PCA, F)")
print(f"IVA channel patterns   : {iva_components.shape}  (S, N_PCA, C)")

# The per-condition split, for reference — no contrast is computed here.
for condition in pooled.conditions:
    print(
        f"  {condition.value:<12}: sources "
        f"{pooled.condition_subjects(iva_sources, condition).shape}, patterns "
        f"{pooled.condition_subjects(iva_components, condition).shape}"
    )

---
## Step 5 — Components arrive with an arbitrary per-recording sign

`iva_g` fixes each component only up to a per-dataset sign, so recording *i*'s copy of
component *k* may be the negation of recording *j*'s. Averaging unresolved maps drives the
group mean toward zero, and on a pooled dataset it is worse than a lost mean: the flips
fall arbitrarily across the two condition blocks, so an unresolved sign manufactures a
condition difference out of nothing.

Only bookkeeping happens here — the sign itself is resolved in **Step 5b**, against the
ASSR electrode topography. An earlier version of this notebook resolved it here from PC1
of the TF maps instead; that made a component's sign *consistent* in the sense PC1
defines, which is not the same as "positive means more power over the ASSR area" — and
the latter is what every contrast and figure downstream actually needs. Anchoring
directly to the topography delivers it in one step, so the PC1 pass has been dropped
rather than composed with it.


In [ ]:
# Which components the comparison figures draw. The SIGN is not decided here — Step 5b
# anchors it to the ASSR electrodes, which is the only alignment this notebook applies.
COMP_INDICES = (
    list(range(N_COMPONENTS_PCA)) if COMPONENTS_TO_PLOT is None else COMPONENTS_TO_PLOT
)
print(f"Components in the comparison figures: {[k + 1 for k in COMP_INDICES]}")
print(
    f"Signs still unresolved at this point: {n_subjects} recordings x "
    f"{N_COMPONENTS_PCA} components, each +-1 as iva_g happened to return it."
)


---
## Step 6 — Condition-Mean Comparison per Component

The overview: one grid per quantity, **rows = conditions, columns = components**, so
every component's Placebo and Psilocybin means sit one above the other. With two
conditions a **difference** row (Psilocybin − Placebo) is added, because a small
difference between two similar maps is far easier to see drawn directly than inferred
from the two panels above it.

Scaling rules, and why:

- **Each column has one symmetric limit shared by the condition rows.** A comparison
  drawn on two independent scales is not a comparison.
- **Limits are not shared across columns.** IVA fixes each component's scale
  independently, so a common limit would render the weaker components flat. Each
  column's `|max|` is printed in its title, and for that reason the grid carries no
  single colourbar — with a limit per column it would be meaningless.
- **The difference row gets its own limit per column**, since it is much weaker than the
  maps it comes from.
- **Every participant is put on a common scale first**
  (`equalize_subject_influence`). The per-recording gain IVA leaves behind is a
  nuisance; without removing it the loudest few participants set the colour limit *and*
  dominate both condition means — precisely the quantity being compared.

Topographies are the **forward (mixing) patterns** recovered in Step 4, on the notebook's
channel subset, so the topomap `Info` is restricted to the same channels in the same
order.

In [ ]:
# Topomap layout: the analyser's Info, restricted to the IVA channel subset. Both
# conditions were preprocessed onto the same montage, so either analyser will do.
iva_info = topo_info_subset(
    condition_analyzers[pooled.conditions[0]].info, n_channels
)
iva_ch_names = list(iva_info["ch_names"])
print(f"Topomap channels : {len(iva_ch_names)} "
      f"(first={iva_ch_names[0]}, last={iva_ch_names[-1]})")

# Row order and the per-recording bookkeeping the figures index by.
CONDITION_ROWS = [c.value for c in pooled.conditions]
tf_time_marks = (
    stimulus_onset_times if MARK_STIMULUS_ONSETS_ON_TF else None
)

fig_topo = plot_condition_mean_topomaps(
    iva_components,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    iva_info,
    n_channels,
    COMP_INDICES,
    label=LABEL,
    alignment_note=ALIGNMENT_NOTE,
    save_path=(PLOTS_DIR / "condition_mean_topomaps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

fig_tf = plot_condition_mean_tf_maps(
    iva_sources,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    FREQS,
    time,
    COMP_INDICES,
    label=LABEL,
    time_marks=tf_time_marks,
    freq_marks=TF_FREQ_MARKS,
    alignment_note=ALIGNMENT_NOTE,
    save_path=(PLOTS_DIR / "condition_mean_tf_maps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

if SAVE_PLOTS:
    print(f"Saved condition-mean figures to {PLOTS_DIR}")

---
## Step 5b — Anchor every component's sign to the ASSR electrodes

The alignment above makes a component's sign **consistent** across recordings, in the sense
PC1 of the TF maps defines. That is not the same as "positive means more power over the
ASSR area", and it is the second property every downstream figure and test depends on:
without it a positive value means *more* power there for some recordings and *less* for
others, and a group mean partly cancels.

This step fixes that, per **(recording, component)** — each component carries its own
independent sign, so one flip shared across a recording's components would be wrong.

The anchor is `corr(topography, 0/1 ASSR mask)` taken across the whole channel axis.
Since `cov(pattern, mask) = f(1-f)(mean_inside - mean_outside)` for a binary mask, that
asks whether the pattern is more positive over the ASSR area **than over the rest of the
head** — so a pattern riding on a global offset cannot flip it, which an anchor reading
only the mean *inside* the mask cannot guarantee. It reads the topography alone and
never the response, so it stays symmetric in the conditions and cannot manufacture a
contrast.

Every pair is flipped on the sign of that correlation however small it is: declining to
flip a weak one does not avoid a choice, it just keeps the run's arbitrary sign instead.
The weak ones earn a printed count, because a wrongly flipped recording *cancels* signal in
a group mean rather than merely widening it.

This matches what `scripts/run_iva_condition_comparison.py` writes into the store, so the
figures below are oriented the same way as the arrays every later analysis reads.


In [ ]:
# The 0/1 electrode selection, on this run's own channel axis.
ANCHOR_COORDINATE_SYSTEM = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
ANCHOR_MASK_STRICT = True

assr_anchor_mask = assr_electrode_mask(
    list(iva_info["ch_names"]), ANCHOR_COORDINATE_SYSTEM, strict=ANCHOR_MASK_STRICT
)

mask_flip, mask_strength = at.polarity_flip(
    iva_components, assr_anchor_mask
)

# Applied to every quantity carrying the per-(recording, component) sign, exactly
# as the PC1 pass was — topography and map must agree about which way is up.
iva_sources = apply_component_signs(iva_sources, mask_flip)
iva_components = apply_component_signs(iva_components, mask_flip)
iva_scores_pca = iva_sources.reshape(n_subjects, N_COMPONENTS_PCA, n_samples_ft)
iva_time = iva_sources.mean(axis=2)  # (S, N_PCA, T)
iva_freq = iva_sources.mean(axis=3)  # (S, N_PCA, F)

n_weak, n_total = at.polarity_weak_count(mask_strength)
print(
    f"ASSR-mask sign anchor over {int(assr_anchor_mask.sum())} electrode(s): "
    f"{int((mask_flip < 0).sum())}/{mask_flip.size} (recording, component) "
    "pair(s) flipped."
)
print("\n  IC   flipped   median |corr|   weak")
for k in range(N_COMPONENTS_PCA):
    weak_k = int((mask_strength[:, k] < at.POLARITY_CORR_FLOOR).sum())
    print(
        f"  {k + 1:>3}   {int((mask_flip[:, k] < 0).sum()):>7}   "
        f"{np.median(mask_strength[:, k]):>13.3f}   "
        f"{f'{weak_k}/{mask_flip.shape[0]}':>4}"
    )
print(
    f"\n  {n_weak}/{n_total} pair(s) decided on |corr| < "
    f"{at.POLARITY_CORR_FLOOR}. Still flipped — the alternative is the run's\n"
    "  own arbitrary sign, not a safer one — but a wrongly flipped recording\n"
    "  CANCELS signal in a group mean rather than merely widening it."
)

# The note every figure below prints, now naming the anchor actually in force.
ALIGNMENT_NOTE = "corr(topography, ASSR electrode mask), per (recording, component)"


---
## Step 7 — Per-Participant Comparison per Component

One figure per component, **Placebo on the first row and Psilocybin on the second**, one
column per participant, columns ordered by **participant ID**. So a participant's two
recordings sit directly one above the other, and the same participant occupies the same
column in every figure.

That vertical pairing is the whole point. The design is within-participant — the pooling
keeps only participants recorded under both conditions — so this turns the contrast into
a within-participant read and answers the question the means cannot: is a difference in
the condition means shared across the group, or carried by one or two people?

Here **every panel of a figure shares one colour limit**, across participants *and*
conditions, so the entire grid is comparable and the single colourbar means something.
A trailing column holds each condition's mean on its own scale — averaging cancels the
incoherent part of every map, so the means read flat under the participants' limit — and
is annotated with its own `|max|`.

Figures are written flat into `PLOTS_DIR`, numbered 1-based in **component order** to
match the `IC <k+1>` labels used above.

In [ ]:
topo_paths = plot_participant_condition_topomaps(
    iva_components,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    iva_info,
    n_channels,
    COMP_INDICES,
    label=LABEL,
    root_dir=PLOTS_DIR / "participants",
    alignment_note=ALIGNMENT_NOTE,
)
tf_paths = plot_participant_condition_tf_maps(
    iva_sources,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    FREQS,
    time,
    COMP_INDICES,
    label=LABEL,
    root_dir=PLOTS_DIR / "participants",
    time_marks=tf_time_marks,
    freq_marks=TF_FREQ_MARKS,
    alignment_note=ALIGNMENT_NOTE,
)
print(f"Wrote {len(topo_paths)} topography figures and {len(tf_paths)} TF figures to "
      f"{PLOTS_DIR / 'participants'}")

# Show the figures inline as well, in component order.
for path in topo_paths + tf_paths:
    display(Image(filename=str(path)))

---
## Step 8 — Stimulus-Locked TF Comparison (Averaged Over Stimuli)

Step 6 and Step 7 draw each component's TF map over the **whole** recording. That view
answers "what does this component do", but it cannot answer "what does it do *to a
stimulus*": the ASSR is a train of short stimuli, and a response locked to their onsets
is smeared across the whole map when the map spans every stimulus at once.

So this is the third view, and the one the 05 quality workflow is built on: **cut a fixed
epoch around every stimulus onset and average it**, per recording and per component. What
was a `(F, T)` whole-recording map becomes a `(F, W)` onset-averaged map, and the
condition contrast is then a contrast of stimulus responses.

The epoch is the **paradigm window** from
[`AssrEpoch`](../../src/definitions/constants.py), via `iva_quality.onset_window` — the
same call the 05 notebook and the headless `--quality` path make, so no onset-locked ASSR
figure in this project is cut on a different window than another:

- a **short interval before each onset** (`EPOCH_PRE_S` = 0.1 s), which is the baseline;
- then `EPOCH_POST_S` = 1.0 s after it — the 0.5 s stimulus plus 0.5 s of post-stimulus,
  so a response that outlasts the stimulus is still visible;
- capped by the shortest inter-onset gap, so an epoch can never reach the next stimulus.

**No baseline subtraction, deliberately.** `zscore_by_time` (Step 1) already zeroed each
`(recording, channel, frequency)` series' time-mean, so the pre-onset interval reads ≈ 0
by construction and subtracting it would only add noise. This is the same convention as
the 05 quality notebook, whose boxcar reference is `0` before the onset for exactly this
reason. The pre-onset interval is therefore worth *looking* at: it is where the map should
be flat, so structure there is a warning sign, not a response.

Both panels of Step 6 and Step 7 are redrawn on this epoch, with the same scaling rules.
The prominent black lines are the stimulus **onset** (dashed, `t = 0`) and its **offset**
(dotted); the green line is the 40 Hz stimulation frequency. Keeping the post-offset tail
in view is what lets a response that stops with the stimulus be told apart from one that
runs on.

Skipped for experiments without stimulus annotations (e.g. PSILO_MUSIC), and for a time
subset too short to fit `MIN_ONSETS_FOR_EPOCH_AVERAGE` epochs.

In [ ]:
onset_samples_in = (
    np.array([], dtype=int)
    if _onset_samples is None
    else _onset_samples[_onset_samples < n_times].astype(int)
)

# The paradigm window, capped by the shortest gap so epochs never overlap.
if onset_samples_in.size:
    EPOCH_PRE, EPOCH_POST = iva_quality.onset_window(onset_samples_in, n_times, sfreq)
    # How many of those epochs actually fit inside the (possibly subset) time window.
    n_fitting = int(
        (
            (onset_samples_in - EPOCH_PRE >= 0)
            & (onset_samples_in + EPOCH_POST <= n_times)
        ).sum()
    )
else:
    EPOCH_PRE = EPOCH_POST = 0
    n_fitting = 0

run_epoch_comparison = n_fitting >= MIN_ONSETS_FOR_EPOCH_AVERAGE
if not run_epoch_comparison:
    print(
        f"Stimulus-locked comparison skipped: {n_fitting} epoch(s) fit the "
        f"{n_times}-sample window, need {MIN_ONSETS_FOR_EPOCH_AVERAGE}. "
        "Raise N_TIMES_SUBSET (or use an experiment with stimulus annotations)."
    )
else:
    # Average the fixed window around every onset: (S, K, F, T) -> (S, K, F, W).
    onset_tf, n_used = iva_quality.epoch_average(
        iva_sources, onset_samples_in, EPOCH_PRE, EPOCH_POST
    )
    epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq  # t=0 at the onset
    # Reference every FREQUENCY to its own pre-onset mean. The z-scoring puts the
    # pre-onset interval near 0 over the WHOLE recording, but not within any single
    # epoch — the local level still drifts — so this is what turns the map into a change
    # rather than a level. Mean only, no divisor: 1/f is already gone (the data were
    # z-scored per channel-frequency before the decomposition), and at the bottom of the
    # map a 100 ms baseline spans under one cycle, so its SD is mostly wavelet phase and
    # dividing by it would manufacture texture there.
    onset_tf = iva_quality.subtract_epoch_baseline(onset_tf, epoch_times < 0.0)
    # Onset (dashed) and stimulus offset (dotted), clamped to the epoch so the offset
    # line is never drawn outside it.
    stimulus_offset_s = min(
        AssrEpoch.STIMULUS_DURATION_S, float(epoch_times[-1])
    )
    EPOCH_MARKS = [0.0, stimulus_offset_s]

    print(f"Onset-averaged TF maps : {onset_tf.shape}  (S, N_PCA, F, W)")
    print(f"Epochs averaged        : {n_used} of {len(onset_samples_in)} onset(s) "
          f"in the window")
    print(f"Epoch window           : {EPOCH_PRE + EPOCH_POST} samples "
          f"({EPOCH_PRE} pre, {EPOCH_POST} post) = "
          f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
    if EPOCH_POST < int(round(EPOCH_POST_S * sfreq)):
        print(f"  NOTE: post-onset span trimmed from {EPOCH_POST_S} s to "
              f"{EPOCH_POST / sfreq:.3f} s by the shortest inter-onset gap.")
    # The baseline should read ~0: zscore_by_time zeroed each series' time-mean, so
    # structure before the onset is a warning sign rather than a response.
    _baseline = np.abs(onset_tf[:, COMP_INDICES, :, :EPOCH_PRE]).mean()
    _response = np.abs(onset_tf[:, COMP_INDICES, :, EPOCH_PRE:]).mean()
    print(f"Mean |baseline| / mean |post-onset| : "
          f"{_baseline:.4g} / {_response:.4g}  (ratio {_baseline / _response:.2f})")

### Condition means — onset-averaged

Same grid as Step 6, on the stimulus-locked epoch.

In [ ]:
if run_epoch_comparison:
    fig_tf_onset = plot_condition_mean_tf_maps(
        onset_tf,
        subject_participants,
        subject_conditions,
        CONDITION_ROWS,
        FREQS,
        epoch_times,
        COMP_INDICES,
        label=LABEL,
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        alignment_note=ALIGNMENT_NOTE,
        save_path=(
            (PLOTS_DIR / "condition_mean_tf_maps_onset.png") if SAVE_PLOTS else None
        ),
    )
    plt.show()
    plt.close("all")

### Per participant — onset-averaged

Same layout as Step 7 — Placebo first row, Psilocybin second, columns by participant ID
— on the stimulus-locked epoch. Written with an `onset_` filename prefix so these sit
alongside the whole-recording figures rather than replacing them.

In [ ]:
if run_epoch_comparison:
    onset_tf_paths = plot_participant_condition_tf_maps(
        onset_tf,
        subject_participants,
        subject_conditions,
        CONDITION_ROWS,
        FREQS,
        epoch_times,
        COMP_INDICES,
        label=LABEL,
        root_dir=PLOTS_DIR / "participants",
        prefix="onset_",
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        alignment_note=ALIGNMENT_NOTE,
    )
    print(f"Wrote {len(onset_tf_paths)} onset-averaged TF figures to "
          f"{PLOTS_DIR / 'participants'}")
    for path in onset_tf_paths:
        display(Image(filename=str(path)))